# Interactive Hand Play

This notebook allows you to play a full hand interactively in Jupyter.

You can:
- View your hand
- Make decisions (order up, call trump, discard)
- Play tricks
- See hand results


In [1]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

from eucher.cards import Card, Deck, Suit
from eucher.game import Game
from eucher.rules import RulesEngine
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from typing import List, Optional


In [2]:
class InteractiveHand:
    """Interactive hand play interface."""
    
    def __init__(self):
        self.deck = Deck()
        self.deck.shuffle()
        self.hand = self.deck.deal(5)
        self.turned_card = self.deck.draw_one()
        self.trump_suit: Optional[Suit] = None
        self.dealer_id = 3  # Player 3 is dealer
        self.player_id = 0  # You are player 0
        self.team = 0  # You are on team 0
        self.tricks_won = [0, 0]
        self.current_trick_cards: List[Card] = []
        self.current_trick_player_ids: List[int] = []
        self.led_suit: Optional[Suit] = None
        self.rules = RulesEngine()
        self.hand_complete = False
        
    def display_state(self) -> None:
        """Display current hand state."""
        html = f"""
        <div style="border: 2px solid #333; padding: 10px; margin: 10px;">
            <h3>Interactive Hand</h3>
            <p><strong>Your Hand:</strong> {', '.join(str(c) for c in self.hand)}</p>
            <p><strong>Turned Card:</strong> {self.turned_card}</p>
            <p><strong>Trump Suit:</strong> {self.trump_suit.name if self.trump_suit else 'Not Set'}</p>
            <p><strong>Tricks Won:</strong> Team 0: {self.tricks_won[0]}, Team 1: {self.tricks_won[1]}</p>
            <p><strong>Current Trick:</strong> {', '.join(str(c) for c in self.current_trick_cards) if self.current_trick_cards else 'Not Started'}</p>
        </div>
        """
        display(HTML(html))

# Initialize hand
hand = InteractiveHand()
hand.display_state()


In [3]:
# Order up decision
if hand.trump_suit is None:
    order_up_button = widgets.Button(description="Order Up", button_style='success')
    pass_button = widgets.Button(description="Pass", button_style='warning')
    
    def on_order_up(b):
        hand.trump_suit = hand.turned_card.suit
        clear_output(wait=True)
        hand.display_state()
        display(HTML("<p>You ordered up! Trump is now " + hand.trump_suit.name + "</p>"))
    
    def on_pass(b):
        clear_output(wait=True)
        hand.display_state()
        display(HTML("<p>You passed. Waiting for other players...</p>"))
        # In a real game, other players would decide here
        # For now, assume someone else calls trump
        hand.trump_suit = hand.turned_card.suit
        hand.display_state()
    
    order_up_button.on_click(on_order_up)
    pass_button.on_click(on_pass)
    
    display(widgets.HBox([order_up_button, pass_button]))


## Discard Decision

In [4]:
# After ordering up, discard a card
if hand.trump_suit and len(hand.hand) == 6:
    discard_buttons = []
    for card in hand.hand:
        if card != hand.turned_card:  # Can't discard the turned card
            button = widgets.Button(description=f"Discard {card}", button_style='info')
            def make_discard(card_to_discard):
                def on_discard(b):
                    hand.hand.remove(card_to_discard)
                    clear_output(wait=True)
                    hand.display_state()
                    display(HTML(f"<p>Discarded {card_to_discard}</p>"))
                return on_discard
            button.on_click(make_discard(card))
            discard_buttons.append(button)
    
    if discard_buttons:
        display(widgets.VBox(discard_buttons))


## Play Tricks

In [5]:
# Play tricks
def create_trick_card_buttons():
    """Create buttons for playing cards in a trick."""
    valid_cards = hand.rules.get_valid_plays(hand.hand, hand.led_suit, hand.trump_suit)
    
    buttons = []
    for card in valid_cards:
        button = widgets.Button(description=str(card), button_style='success')
        
        def make_play_card(card_to_play):
            def on_button_click(b):
                try:
                    hand.hand.remove(card_to_play)
                    hand.current_trick_cards.append(card_to_play)
                    hand.current_trick_player_ids.append(hand.player_id)
                    
                    if len(hand.current_trick_cards) == 1:
                        hand.led_suit = card_to_play.suit
                    
                    clear_output(wait=True)
                    hand.display_state()
                    
                    if len(hand.current_trick_cards) < 4:
                        # Simulate other players (simplified)
                        for _ in range(4 - len(hand.current_trick_cards)):
                            # Add placeholder cards for other players
                            hand.current_trick_cards.append(Card(Suit.HEARTS, Rank.TEN))
                            hand.current_trick_player_ids.append(len(hand.current_trick_player_ids))
                        
                        # Determine winner
                        winner_id = hand.rules.determine_trick_winner(
                            hand.current_trick_cards,
                            hand.current_trick_player_ids,
                            hand.led_suit,
                            hand.trump_suit
                        )
                        winner_team = winner_id % 2
                        hand.tricks_won[winner_team] += 1
                        
                        display(HTML(f"<h3>Trick Complete! Winner: Player {winner_id} (Team {winner_team})</h3>"))
                        display(HTML(f"<p>Tricks: Team 0 = {hand.tricks_won[0]}, Team 1 = {hand.tricks_won[1]}</p>"))
                        
                        # Reset for next trick
                        hand.current_trick_cards = []
                        hand.current_trick_player_ids = []
                        hand.led_suit = None
                        
                        if sum(hand.tricks_won) >= 5:
                            hand.hand_complete = True
                            display(HTML("<h2>Hand Complete!</h2>"))
                        else:
                            display(HTML("<p>Starting next trick...</p>"))
                            display(create_trick_card_buttons())
                    else:
                        display(create_trick_card_buttons())
                except Exception as e:
                    display(HTML(f"<p style='color: red;'>{e}</p>"))
                    hand.display_state()
                    display(create_trick_card_buttons())
            
            return on_button_click
        
        button.on_click(make_play_card(card))
        buttons.append(button)
    
    return widgets.VBox(buttons)

# Display trick playing interface
if not hand.hand_complete and hand.trump_suit and len(hand.hand) == 5:
    display(create_trick_card_buttons())
